# Решения: stack, queue и deque

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from collections import deque


## Урок. 1. Поток событий

In [ ]:
events = [(r.order_id, "late_review" if r.is_late else "standard") for r in df.head(12).itertuples()]
assert len(events) == min(12, len(df))
print(events[:3])


## Урок. 2–3. Stack и undo

In [ ]:
steps = ["load", "drop_missing", "make_delay", "scale"]
history = []
for step in steps:
    history.append(step)
undone = history.pop()

def undo(stack):
    return stack.pop() if stack else None

trial = ["load", "filter"]
removed = undo(trial)
assert undone == "scale" and removed == "filter" and undo([]) is None


## Урок. 4. FIFO queue

In [ ]:
queue = deque(item[0] for item in events)
processed = [queue.popleft() for _ in range(min(3, len(queue)))]
assert processed == [item[0] for item in events[:3]]


## Урок. 5. Срочное событие

In [ ]:
normal_ids = df.loc[df["is_late"].eq(0), "order_id"].head(5).tolist()
late_id = df.loc[df["is_late"].eq(1), "order_id"].iloc[0]
dispatch = deque(normal_ids)
dispatch.appendleft(late_id)
next_id = dispatch.popleft()
assert next_id == late_id and list(dispatch) == normal_ids


## Урок. 6. Ограниченный буфер

In [ ]:
recent = deque(maxlen=5)
for oid in df["order_id"]:
    recent.append(oid)
snapshot = list(recent)
assert snapshot == df["order_id"].tail(min(5, len(df))).tolist()


## Урок. 7. Семантика операций

In [ ]:
choices = {"undo_preprocessing": "stack", "fifo_orders": "queue", "urgent_both_ends": "deque"}
assert set(choices.values()) == {"stack", "queue", "deque"}


## Урок. 8. Два порядка

In [ ]:
q_left = deque(item[0] for item in events)
q_right = deque(item[0] for item in events)
left_order = [q_left.popleft() for _ in range(len(q_left))]
right_order = [q_right.pop() for _ in range(len(q_right))]
ORDER_NOTE = "popleft сохраняет FIFO: первым обработан первый пришедший заказ; pop справа разворачивает поток и меняет смысл очереди."
assert right_order == list(reversed(left_order)) and len(ORDER_NOTE) >= 80


## Урок. 9. Журнал

In [ ]:
pipeline = ["load", "clean", "join", "scale", "cluster"]
history = pipeline.copy()
undone_two = [history.pop(), history.pop()]
audit = {"remaining": history, "undone": undone_two}
assert audit["remaining"] == pipeline[:3]


## ДЗ. A1–A3

In [ ]:
ops = ["load", "clean", "scale", "cluster", "report"]
stack = ops.copy()
undo_order = [stack.pop() for _ in range(len(stack))]
source_ids = df["order_id"].head(10).tolist()
q = deque(source_ids)
fifo = [q.popleft() for _ in range(len(q))]
late_ids = df.loc[df["is_late"].eq(1), "order_id"].head(4).tolist()
normal_ids = df.loc[df["is_late"].eq(0), "order_id"].head(4).tolist()
priority_order = late_ids + normal_ids
assert undo_order == list(reversed(ops)) and fifo == source_ids


## ДЗ. Challenge

In [ ]:
def dispatch_orders(late, normal, limit):
    late, normal = deque(late), deque(normal)
    result = []
    while len(result) < limit and (late or normal):
        result.append(late.popleft() if late else normal.popleft())
    return result

result = dispatch_orders(deque(late_ids), deque(normal_ids), 5)
STRUCTURE_NOTE = (
    "Stack задаёт LIFO и подходит для undo. Queue задаёт FIFO и сохраняет порядок потока. "
    "Deque поддерживает быстрые операции с обоих концов: обычные события справа, срочные слева. "
    "Выбор определяется смыслом операции, а не названием контейнера."
)
assert result == (late_ids + normal_ids)[:5] and len(STRUCTURE_NOTE) >= 180
